<a href="https://colab.research.google.com/github/BardRimon/Study/blob/main/CV/HW3_ImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее  задание
Ознакомиться с одной из научных статей (по вариантам):

* [Physiological Inspired Deep Neural Networks for Emotion Recognition](https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=8472816&tag=1)

* [Boundary loss for highly unbalanced segmentation](https://arxiv.org/abs/1812.07032)


* [Correlation Maximized Structural Similarity Loss for Semantic Segmentation](https://arxiv.org/abs/1910.08711)

* [Topology-Preserving Deep Image Segmentation](https://papers.nips.cc/paper/8803-topology-preserving-deep-image-segmentation)



На основе выбранной статьи:

1. реализовать функцию потерь которая описана в статье,
2. описать её математическую формулу и интуицию работы,
3. объяснить, какие проблемы она решает (например, дисбаланс классов, сохранение границ, сохранение топологии и т. д.).
4. реализовать одну модель сегментации:
(**варианты: LinkNet, U-Net, DeepLab v1, PSPNet**);
5. провести обучение и сравнение результатов с базовыми функциями потерь
(BCE, Dice, Focal, Tversky) на одном и том же датасете.
6. Создать гибридную функцию потерь —
объединить предложенный лосс с одной из классических (например, Boundary + Dice, SSIM + Focal) и сравнить динамику сходимости и итоговые метрики (IoU, Dice, Precision, Recall).


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PhysiologicalEmotionLoss(nn.Module):
    """
    Implementation of the loss function described in 'Physiological Inspired Deep Neural Networks
    for Emotion Recognition'.

    This loss includes:
    1. [cite_start]Classification Loss (Categorical Cross-Entropy) [cite: 217]
    2. Facial Parts Loss, which can be:
       - [cite_start]Fully Supervised (MSE with target maps) [cite: 228]
       - [cite_start]Weakly Supervised (Sparsity + Spatial Contiguity) [cite: 247]
       - [cite_start]Hybrid (Combination of both) [cite: 272]
    """

    def __init__(self,
                 lambda_main=1.0,
                 lambda_sparsity=1e-4,
                 gamma_contiguity=1.0,
                 mode='hybrid'):
        """
        Args:
            lambda_main (float): Weight λ controlling interaction between classification
                                 [cite_start]and facial parts loss (Eq. 2)[cite: 216].
            [cite_start]lambda_sparsity (float): Weight λ for the sparsity term in weakly supervised mode (Eq. 5)[cite: 364].
            [cite_start]gamma_contiguity (float): Weight γ for the contiguity term in weakly supervised mode (Eq. 5)[cite: 257].
            mode (str): One of 'supervised', 'weakly', or 'hybrid'.
        """
        super(PhysiologicalEmotionLoss, self).__init__()
        self.lambda_main = lambda_main
        self.lambda_sparsity = lambda_sparsity
        self.gamma_contiguity = gamma_contiguity
        self.mode = mode

        # [cite_start]Classification loss is Categorical Cross-Entropy (Eq. 3) [cite: 217]
        self.classification_criterion = nn.CrossEntropyLoss()

        # [cite_start]Fully supervised loss is Mean Squared Error (Eq. 4) [cite: 243]
        self.mse_criterion = nn.MSELoss()

    def _sparsity_loss(self, pred_map):
        """
        [cite_start]Calculates the sparsity term (L1 regularization) defined in Eq. 6[cite: 260].
        L_sparsity = (1 / m*n) * sum(|x_ij|)
        """
        # Calculate mean of absolute values (equivalent to sum divided by m*n pixels)
        return torch.mean(torch.abs(pred_map))

    def _contiguity_loss(self, pred_map):
        """
        Calculates the spatial contiguity term (Total Variation) defined in Eq. [cite_start]7[cite: 266].
        Minimizes local spatial transitions to encourage smoothness.
        """
        # pred_map shape: (batch, channels, height, width)
        h, w = pred_map.shape[2], pred_map.shape[3]

        # Calculate differences between adjacent pixels in height and width dimensions
        diff_h = torch.abs(pred_map[:, :, 1:, :] - pred_map[:, :, :-1, :])
        diff_w = torch.abs(pred_map[:, :, :, 1:] - pred_map[:, :, :, :-1])

        # Sum of differences normalized by image resolution (m * n)
        # Note: We sum over the spatial dimensions and divide by total pixels
        loss_h = torch.sum(diff_h) / (h * w * pred_map.shape[0] * pred_map.shape[1])
        loss_w = torch.sum(diff_w) / (h * w * pred_map.shape[0] * pred_map.shape[1])

        return loss_h + loss_w

    def forward(self, pred_class, target_class, pred_map, target_map=None):
        """
        Args:
            pred_class (Tensor): Predicted logits for expression classes (y_hat).
            target_class (Tensor): Ground truth class indices.
            [cite_start]pred_map (Tensor): Predicted relevance map (x_hat) from Facial Parts Component[cite: 165].
            target_map (Tensor, optional): Target relevance map (x_target) generated from landmarks.
                                           [cite_start]Required for 'supervised' and 'hybrid' modes[cite: 239].
        """
        # [cite_start]1. Classification Loss (Eq. 3) [cite: 219]
        loss_classification = self.classification_criterion(pred_class, target_class)

        loss_facial_parts = 0.0

        # 2. Facial Parts Loss
        # [cite_start]Fully Supervised Component (Eq. 4) [cite: 243]
        if self.mode in ['supervised', 'hybrid']:
            if target_map is None:
                raise ValueError("target_map cannot be None for supervised or hybrid mode.")
            loss_supervised = self.mse_criterion(pred_map, target_map)
            loss_facial_parts += loss_supervised

        # [cite_start]Weakly Supervised Component (Eq. 5, 6, 7) [cite: 249, 260, 266]
        if self.mode in ['weakly', 'hybrid']:
            l_sparsity = self._sparsity_loss(pred_map)
            l_contiguity = self._contiguity_loss(pred_map)

            # [cite_start]Weighted sum for weakly supervised part (Eq. 5) [cite: 256]
            loss_weakly = (self.lambda_sparsity * l_sparsity) + (self.gamma_contiguity * l_contiguity)

            # If hybrid, we combine them. [cite_start]The paper suggests weighted summation (Section III-B-3)[cite: 274].
            # Assuming simple addition logic based on "weighted summation of loss terms defined in Eq 4 and 5"
            loss_facial_parts += loss_weakly

        # [cite_start]3. Total Loss (Eq. 2) [cite: 214]
        # L = L_classification + lambda * L_facial_parts
        total_loss = loss_classification + (self.lambda_main * loss_facial_parts)

        return total_loss

### Общая формулировка
Функция потерь $L$ является составной. Она обучает модель одновременно классифицировать эмоции и определять «карту релевантности» (relevance map), которая подсвечивает важные участки лица.

Формула выглядит следующим образом:
$$L = \mathcal{L}_{classification} + \lambda \mathcal{L}_{facial\_parts}$$
где $\lambda \ge 0$ — гиперпараметр, контролирующий баланс между задачей классификации и задачей выделения признаков лица.

---

### 1. Компонент классификации ($\mathcal{L}_{classification}$)
**Формула:** Используется стандартная категориальная кросс-энтропия (Categorical Cross-Entropy):
$$\mathcal{L}_{classification} = -\sum_{i=1}^{N} y_i \log(\hat{y}_i)$$
где $y_i$ — истинная метка класса (one-hot вектор), а $\hat{y}_i$ — предсказанная вероятность, полученная после softmax.

**Интуиция:** Этот член заставляет сеть предсказывать правильную эмоцию (например, радость или злость) на основе признаков, извлеченных из изображения.

---

### 2. Компонент лицевых частей ($\mathcal{L}_{facial\_parts}$)
Этот компонент отвечает за обучение «карты релевантности» $\hat{x}$. Его цель — заставить сеть фокусироваться только на тех участках лица, которые физиологически участвуют в выражении эмоций (мышцы, глаза, рот).

В зависимости от режима (Fully Supervised, Weakly Supervised или Hybrid), этот компонент рассчитывается по-разному.

#### А. Полностью контролируемый режим (Fully Supervised)
**Формула:** Используется среднеквадратичная ошибка (MSE) между предсказанной картой релевантности $\hat{x}$ и целевой картой $x^{target}$:
$$\mathcal{L}_{facial\_parts} = \frac{1}{N}\sum_{i=1}^{N}(x_{i}^{target}-\hat{x}_{i})^{2}$$
**Интуиция:** Если у нас есть разметка ключевых точек лица (landmarks), мы создаем идеальную «тепловую карту» ($x^{target}$), состоящую из гауссиан вокруг глаз, носа и рта. Мы прямо учим сеть активироваться именно в этих зонах.

#### Б. Слабо контролируемый режим (Weakly Supervised)
Этот режим используется, когда нет разметки ключевых точек. Он опирается на физиологическую гипотезу о том, что важные регионы лица — это небольшие (разреженные) и цельные (непрерывные) области.

**Формула:**
$$\mathcal{L}_{facial\_parts} = \lambda_{sparsity} \mathcal{L}_{sparsity}(\hat{x}) + \gamma \mathcal{L}_{contiguity}(\hat{x})$$

1.  **Разреженность (Sparsity, $\mathcal{L}_{sparsity}$):**
    * *Математика:* L1-регуляризация активаций карты.
  * $\mathcal{L}_{sparsity} \hat{x}=\frac{1}{m \times n}\sum_{i,j}|\hat{x}_{i,j}|$
    * *Интуиция:* Большинство пикселей на лице не важны для эмоции. Важны только конкретные мышцы. L1-норма заставляет карту быть «пустой» (нули) почти везде, кроме самых важных мест.

2.  **Пространственная непрерывность (Spatial Contiguity, $\mathcal{L}_{contiguity}$):**
    * *Математика:* Total Variation (минимизация разницы между соседними пикселями).
  * $\mathcal{L}_{contiguity}(\hat{x})=\frac{1}{m \times n}\sum |\hat{x}_{i+1,j}-\hat{x}_{i,j}|+|\hat{x}_{i,j+1}-\hat{x}_{i,j}|$


    * *Интуиция:* Активации не должны быть шумными (разбросанными пикселями). Мышцы — это сплошные объекты. Этот член сглаживает карту, заставляя активные регионы быть локализованными пятнами, а не шумом.

#### В. Гибридный режим (Hybrid)
**Формула:** Взвешенная сумма полностью контролируемого и слабо контролируемого методов.
**Интуиция:** Комбинирует сильные стороны обоих подходов. MSE гарантирует, что сеть найдет основные черты (глаза, рот), а слабый контроль (разреженность + непрерывность) позволяет сети самой найти дополнительные важные детали, такие как морщины от эмоций или ямочки, которые не отмечены в стандартных landmarks.

### Итоговая суть (Summary)
Aвторы предлагают архитектуру, которая не просто классифицирует картинку целиком, а сначала учится понимать, *куда* смотреть (), и использует эту информацию для усиления признаков в нужных местах перед классификацией.

## Основные проблемы FER (face emotion recognition)

На момент публикации статьи - малый размер датасета. При tranfer learning модель выдавала низкий скор, т.к. захватывала большое количество "ненужных признаков"

с помощью $\mathcal{L}_{facial\_parts}$ Делался акцент не тех участках, которые напрямую должны были влиять на определение эмоции конкретного сэмпла.

In [2]:
dataset_dir = '/content/PH2Dataset'
output_dir = '/content/'
path_to_dummy_samples = '/content/for_asserts/'
ckpt_path = '/content/ckpt/'
!mkdir /content/for_asserts
!mkdir /content/ckpt

In [3]:
!gdown 1T_RPkPP0jeWwK8L1UrmBw8V30eD7v6Ql
get_ipython().system_raw("unrar x PH2Dataset.rar")

Downloading...
From (original): https://drive.google.com/uc?id=1T_RPkPP0jeWwK8L1UrmBw8V30eD7v6Ql
From (redirected): https://drive.google.com/uc?id=1T_RPkPP0jeWwK8L1UrmBw8V30eD7v6Ql&confirm=t&uuid=86be2723-1ef6-4120-b7f9-fd8ed3429a56
To: /content/PH2Dataset.rar
100% 162M/162M [00:04<00:00, 36.2MB/s]


In [4]:
images = []
lesions = []
from skimage.io import imread
import os
root = 'PH2Dataset'

for root, dirs, files in os.walk(dataset_dir):
    if root.endswith('_Dermoscopic_Image'):
        images.append(imread(os.path.join(root, files[0])))
    if root.endswith('_lesion'):
        lesions.append(imread(os.path.join(root, files[0])))

In [5]:
from skimage.transform import resize
size = (256, 256)
X = [resize(x, size, mode='constant', anti_aliasing=True,) for x in images]
Y = [resize(y, size, mode='constant', anti_aliasing=False) > 0.5 for y in lesions]

In [6]:
import numpy as np
X = np.array(X, np.float32)
Y = np.array(Y, np.float32)
print(f'Loaded {len(X)} images')

Loaded 200 images


In [7]:
ix = np.random.choice(len(X), len(X), False)
tr, val, ts = np.split(ix, [120, 160])

In [8]:
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
batch_size = 10

train_dataloader = DataLoader(list(zip(np.rollaxis(X[tr], 3, 1), Y[tr, np.newaxis])),
                     batch_size=batch_size, shuffle=True, num_workers = 2)
valid_dataloader = DataLoader(list(zip(np.rollaxis(X[val], 3, 1), Y[val, np.newaxis])),
                      batch_size=batch_size, shuffle=False, num_workers = 2)
test_dataloader = DataLoader(list(zip(np.rollaxis(X[ts], 3, 1), Y[ts, np.newaxis])),
                     batch_size=batch_size, shuffle=False, num_workers = 2)

In [10]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# Настройка устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [15]:
class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)

        self.downsample = None
        if stride != 1 or in_planes != out_planes:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_planes)
            )

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DecoderBlock, self).__init__()
        # 1x1 conv для уменьшения каналов
        self.conv1 = nn.Conv2d(in_channels, in_channels // 4, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels // 4)
        self.relu = nn.ReLU(inplace=True)

        # Transposed conv для апскейлинга
        self.deconv = nn.ConvTranspose2d(in_channels // 4, in_channels // 4, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(in_channels // 4)

        # 1x1 conv для получения нужного числа выходных каналов
        self.conv2 = nn.Conv2d(in_channels // 4, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.deconv(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn3(x)
        x = self.relu(x)
        return x

class LinkNet(nn.Module):
    def __init__(self, n_classes=1):
        super(LinkNet, self).__init__()

        # Encoder (ResNet18-like)
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)

        # Decoder
        self.decoder4 = DecoderBlock(512, 256)
        self.decoder3 = DecoderBlock(256, 128)
        self.decoder2 = DecoderBlock(128, 64)
        self.decoder1 = DecoderBlock(64, 64)

        # Final layers
        self.finaldeconv = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.finalbn = nn.BatchNorm2d(32)
        self.finalconv = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.finalout = nn.Conv2d(32, n_classes, kernel_size=1, padding=0) # Changed kernel_size to 1 to match target size

    def _make_layer(self, block, planes, blocks, stride=1):
        layers = []
        layers.append(block(self.in_planes, planes, stride))
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(block(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        # Encoder
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        e1 = self.layer1(x)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        # Decoder
        # LinkNet uses ADDITION for skip connections
        d4 = self.decoder4(e4) + e3
        d3 = self.decoder3(d4) + e2
        d2 = self.decoder2(d3) + e1
        d1 = self.decoder1(d2)

        f = self.finaldeconv(d1)
        f = self.finalbn(f)
        f = self.relu(f)
        f = self.finalconv(f)
        f = self.relu(f)
        out = self.finalout(f)

        return out

In [12]:
# 1. Dice Loss
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)
        return 1 - dice

# 2. Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

# 3. Tversky Loss
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, smooth=1):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha # False Positives weight
        self.beta = beta   # False Negatives weight
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        TP = (inputs * targets).sum()
        FP = ((1 - targets) * inputs).sum()
        FN = (targets * (1 - inputs)).sum()

        Tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        return 1 - Tversky

# 4. Physiological Loss (из статьи)
class PhysiologicalLoss(nn.Module):
    def __init__(self, lambda_sparsity=1e-4, gamma_contiguity=1e-4):
        super(PhysiologicalLoss, self).__init__()
        self.lambda_sparsity = lambda_sparsity
        self.gamma_contiguity = gamma_contiguity

    def _sparsity(self, x):
        return torch.mean(torch.abs(x)) # L1

    def _contiguity(self, x):
        # Total Variation
        h_diff = torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :]).mean()
        w_diff = torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1]).mean()
        return h_diff + w_diff

    def forward(self, inputs, targets=None):
        # targets не используются для регуляризации, только inputs (карта активации)
        probs = torch.sigmoid(inputs)
        loss = self.lambda_sparsity * self._sparsity(probs) + \
               self.gamma_contiguity * self._contiguity(probs)
        return loss

# 5. Hybrid Loss (Dice + Physiological)
class HybridPhysioDiceLoss(nn.Module):
    def __init__(self, physio_weight=0.1):
        super(HybridPhysioDiceLoss, self).__init__()
        self.dice = DiceLoss()
        self.physio = PhysiologicalLoss()
        self.physio_weight = physio_weight

    def forward(self, inputs, targets):
        l_dice = self.dice(inputs, targets)
        l_physio = self.physio(inputs) # Unsupervised part
        return l_dice + self.physio_weight * l_physio

In [17]:
def calculate_metrics(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    target = target.float()

    pred_flat = pred.view(-1).cpu().numpy()
    target_flat = target.view(-1).cpu().numpy()

    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection

    iou = intersection / (union + 1e-6)
    dice = (2. * intersection) / (pred_flat.sum() + target_flat.sum() + 1e-6)

    # Для Precision/Recall лучше использовать sklearn для корректной обработки деления на ноль
    precision = precision_score(target_flat, pred_flat, zero_division=0)
    recall = recall_score(target_flat, pred_flat, zero_division=0)

    return iou, dice, precision, recall

def train_model(model, train_loader, criterion, optimizer, num_epochs=5):
    model.train()
    history = {'loss': []}

    for epoch in range(num_epochs):
        epoch_loss = 0
        if len(train_loader) == 0: # Added check for empty train_loader
            print(f"Warning: train_loader is empty for epoch {epoch+1}. Skipping training for this epoch.")
            history['loss'].append(0.0) # Append 0 or some indicator if no training occurred
            continue # Skip to the next epoch

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        history['loss'].append(avg_loss)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

    return history

def evaluate_model(model, test_loader):
    model.eval()
    metrics = {'iou': 0, 'dice': 0, 'precision': 0, 'recall': 0}
    steps = 0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)

            iou, dice, prec, rec = calculate_metrics(outputs, masks)
            metrics['iou'] += iou
            metrics['dice'] += dice
            metrics['precision'] += prec
            metrics['recall'] += rec
            steps += 1

    for k in metrics:
        metrics[k] /= steps

    return metrics

# === Основной блок запуска экспериментов ===

# Список лоссов для сравнения
losses = {
    'BCE': nn.BCEWithLogitsLoss(),
    'Dice': DiceLoss(),
    'Focal': FocalLoss(),
    'Tversky': TverskyLoss(alpha=0.7, beta=0.3),
    'Hybrid (Dice+Physio)': HybridPhysioDiceLoss(physio_weight=0.1)
}

results = {}

print("--- Начало сравнения лоссов ---")
for loss_name, criterion in losses.items():
    print(f"\nTraining with {loss_name} Loss...")

    # Реинициализация модели для каждого лосса
    model = LinkNet(n_classes=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # Обучение (малое кол-во эпох для демонстрации)
    history = train_model(model, train_dataloader, criterion, optimizer, num_epochs=10) # Changed train_loader to train_dataloader for consistency

    # Оценка
    metrics = evaluate_model(model, test_dataloader)
    results[loss_name] = {'metrics': metrics, 'history': history}

    print(f"Results for {loss_name}: IoU={metrics['iou']:.4f}, Dice={metrics['dice']:.4f}")

# Вывод итоговой таблицы
print("\n=== Итоговое сравнение ===")
print(f"{'Loss Function':<20} | {'IoU':<8} | {'Dice':<8} | {'Precision':<10} | {'Recall':<8}")
print("-" * 65)
for name, res in results.items():
    m = res['metrics']
    print(f"{name:<20} | {m['iou']:.4f}   | {m['dice']:.4f}   | {m['precision']:.4f}     | {m['recall']:.4f}")

--- Начало сравнения лоссов ---

Training with BCE Loss...
Epoch [1/10], Loss: 0.6824
Epoch [2/10], Loss: 0.6359
Epoch [3/10], Loss: 0.5749
Epoch [4/10], Loss: 0.5043
Epoch [5/10], Loss: 0.4383
Epoch [6/10], Loss: 0.3748
Epoch [7/10], Loss: 0.3308
Epoch [8/10], Loss: 0.2849
Epoch [9/10], Loss: 0.2484
Epoch [10/10], Loss: 0.2317
Results for BCE: IoU=0.7782, Dice=0.8740

Training with Dice Loss...
Epoch [1/10], Loss: 0.5918
Epoch [2/10], Loss: 0.5600
Epoch [3/10], Loss: 0.5317
Epoch [4/10], Loss: 0.4795
Epoch [5/10], Loss: 0.4236
Epoch [6/10], Loss: 0.3673
Epoch [7/10], Loss: 0.3079
Epoch [8/10], Loss: 0.2774
Epoch [9/10], Loss: 0.2244
Epoch [10/10], Loss: 0.1940
Results for Dice: IoU=0.8296, Dice=0.9064

Training with Focal Loss...
Epoch [1/10], Loss: 0.1293
Epoch [2/10], Loss: 0.1126
Epoch [3/10], Loss: 0.0959
Epoch [4/10], Loss: 0.0825
Epoch [5/10], Loss: 0.0728
Epoch [6/10], Loss: 0.0659
Epoch [7/10], Loss: 0.0565
Epoch [8/10], Loss: 0.0503
Epoch [9/10], Loss: 0.0452
Epoch [10/10], L